# Lab 02-02: GAN on Fashion-MNIST (Colab T4 Optimized)

## Objective
Train a basic GAN Model (Fashion-MNIST).

## Optimizations
- **Batch Size**: 128
- **Workers**: 2 (pinned memory)
- **Tqdm**: For progress tracking

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm.auto import tqdm

# --- Configuration ---
DATASET_CHOICE = 'fashion'
EPOCHS = 50 
BATCH_SIZE = 128
NOISE_DIM = 100
LR = 0.0002
SAVE_INTERVAL = 5
IMG_SIZE = 28
CHANNELS = 1

FASHION_LABELS = {
    0: 'T-Shirt', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle Boot'
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

os.makedirs("generated_samples/02", exist_ok=True)
os.makedirs("final_generated_images/02", exist_ok=True)
os.makedirs("models/02", exist_ok=True)

In [ ]:
# --- Step 1: Classifier ---
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_classifier():
    print("Training Classifier...")
    # Standard transform for classifier
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    ds = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    
    model = SimpleClassifier().to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for imgs, lbls in tqdm(dl):
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt.zero_grad()
        out = model(imgs)
        loss = crit(out, lbls)
        loss.backward()
        opt.step()
    
    torch.save(model.state_dict(), "models/02/fashion_classifier.pth")
    return model

if not os.path.exists("models/02/fashion_classifier.pth"):
    eval_classifier = train_classifier()
else:
    eval_classifier = SimpleClassifier().to(device)
    eval_classifier.load_state_dict(torch.load("models/02/fashion_classifier.pth")) 
    eval_classifier.eval()

In [ ]:
# --- Step 2: GAN Setup ---
gan_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=gan_transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

class Generator(nn.Module):
    def __init__(self, noise_dim, img_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),
            nn.Tanh()
        )
    def forward(self, x): return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

gen = Generator(NOISE_DIM, 28*28).to(device)
disc = Discriminator(28*28).to(device)
opt_g = optim.Adam(gen.parameters(), lr=LR)
opt_d = optim.Adam(disc.parameters(), lr=LR)
criterion = nn.BCELoss()
fixed_noise = torch.randn(25, NOISE_DIM).to(device)

In [ ]:
# --- Step 3: Training ---
print("Starting Training...")
for epoch in range(EPOCHS):
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_idx, (real, _) in pbar:
        real = real.view(-1, 784).to(device)
        curr_bs = real.size(0)
        
        # Train Discriminator
        d_loss_real = criterion(disc(real), torch.ones(curr_bs, 1).to(device))
        fake = gen(torch.randn(curr_bs, NOISE_DIM).to(device))
        d_loss_fake = criterion(disc(fake.detach()), torch.zeros(curr_bs, 1).to(device))
        d_loss = (d_loss_real + d_loss_fake) / 2
        
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        
        # Train Generator
        g_loss = criterion(disc(fake), torch.ones(curr_bs, 1).to(device))
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
        
        pbar.set_postfix({'D_loss': f"{d_loss.item():.4f}", 'G_loss': f"{g_loss.item():.4f}"})

    if (epoch+1) % SAVE_INTERVAL == 0:
        with torch.no_grad():
            img = gen(fixed_noise).reshape(-1, 1, 28, 28)
            save_image(img, f"generated_samples/02/epoch_{epoch+1:02d}.png", nrow=5, normalize=True)
print("Training Complete")

In [ ]:
# --- Step 4: Evaluation ---
gen.eval()
with torch.no_grad():
    final_fake = gen(torch.randn(100, NOISE_DIM).to(device)).reshape(-1, 1, 28, 28)
    save_image(final_fake, "final_generated_images/02/final_100_samples.png", nrow=10, normalize=True)
    
    outputs = eval_classifier(final_fake)
    _, predicted = torch.max(outputs.data, 1)
    preds = predicted.cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.hist(preds, bins=range(11), align='left', rwidth=0.8, color='lightgreen', edgecolor='black')
    plt.xticks(range(10), list(FASHION_LABELS.values()), rotation=45)
    plt.title('Distribution of Generated Fashion Items')
    plt.show()
    
    print("Counts:")
    unique, counts = np.unique(preds, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"{FASHION_LABELS[u]}: {c}")